<a href="https://colab.research.google.com/github/SelvaKumaran-G/flyrank-ml-internship3/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SelvaKumaran-G/flyrank-ml-internship3/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Finding #1 — "The Anatomy of Growing Content"

**Claim:** Growing pages average 3.2K words vs 2.3K for declining pages, and are
20% younger (184d vs 230d age), based on 74.8K rising vs 45.6K falling pages.

**My methodology question:** The label here is `trend_direction`, derived from
30-day-vs-prior-30-day impression change — a snapshot comparison, not a
tracked cohort followed over time. That raises a directionality question: does
extra word count *cause* growth, or do pages that are already growing simply
get more editorial investment (more sections added) *because* they're
succeeding? The paper is careful to call this "observational," which I
respect — but I'd ask whether a time-lagged design (word count *before* the
growth window, not measured concurrently) was considered, since that would
separate cause from effect more convincingly.

## Finding #4 — "The Freshness Multiplier"

**Claim:** The 361+ day freshness bucket shows a 283:1 growth-to-decline ratio,
but the paper itself flags this is unstable (283 growing pages vs just 1
declining page in that bucket).

**My methodology question:** The paper's own disclosure here is a model of
honesty — it explicitly excludes this ratio from headline claims. My question
is more about the neighboring `181-360` bucket, which *is* used as a stable
headline number (3.13:1): with what denominator size did that ratio stabilize,
and was a formal minimum-sample-size threshold applied consistently across all
buckets, or was `361+` singled out because its result happened to look
implausible? A disclosed, general rule (e.g. "any bucket with under N declining
pages is excluded") would make the standard fully reproducible rather than
judgment-call-by-judgment-call.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

df = pd.read_csv('/content/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

numeric = ['search_volume','competition','cpc','word_count','char_count','days_with_impressions',
           'days_since_last_update','ctr','avg_position','engagement_rate','scroll_rate',
           'ai_traffic_pct','content_age_days']
categorical = ['competition_level','content_type','main_intent']

pre = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                       ('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical),
])

def evaluate(train_df, test_df, label):
    pipe = Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=2000, random_state=42))])
    pipe.fit(train_df[numeric+categorical], train_df['is_declining_label'])
    probs = pipe.predict_proba(test_df[numeric+categorical])[:, 1]
    test_df = test_df.copy()
    test_df['score'] = probs
    p50 = test_df.sort_values('score', ascending=False).head(50)['is_declining_label'].mean()
    return p50

# BEFORE: naive random row split
train_r, test_r = train_test_split(df, test_size=0.2, random_state=42, stratify=df['is_declining_label'])
overlap_before = len(set(train_r['client_id']) & set(test_r['client_id']))
p_before = evaluate(train_r, test_r, "before")

# AFTER: client-grouped split
splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
tr_idx, te_idx = next(splitter.split(df, groups=df['client_id']))
train_g, test_g = df.iloc[tr_idx], df.iloc[te_idx]
overlap_after = len(set(train_g['client_id']) & set(test_g['client_id']))
p_after = evaluate(train_g, test_g, "after")

print(f"BEFORE (random split):  Precision@50 = {p_before:.3f}  |  client overlap: {overlap_before}/{df['client_id'].nunique()}")
print(f"AFTER  (grouped split): Precision@50 = {p_after:.3f}  |  client overlap: {overlap_after}")
print(f"Optimistic inflation from leakage: {p_before - p_after:.3f}")

BEFORE (random split):  Precision@50 = 0.780  |  client overlap: 31/32
AFTER  (grouped split): Precision@50 = 0.700  |  client overlap: 0
Optimistic inflation from leakage: 0.080


| Split | Precision@50 | Client overlap |
|---|---|---|
| Before (random row split) | ~0.780 | 31 of 32 clients |
| After (client-grouped split) | ~0.700 | 0 |

Before/after gap: about 8 points of Precision@50 were an artifact of client
leakage, not real generalization. A near-total client overlap (31 of 32) in
the naive split let the model partially memorize client-specific patterns
rather than learn signals that transfer to unseen clients. The grouped split
is the honest number I should report and did report in Week 5.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [4]:
# Re-check every feature used against the label definition
label_source = 'trend_direction'  # is_declining_label is derived from this

used_features = numeric + categorical
print("Features used:", used_features)
print()
print(f"Label-derivation check: '{label_source}' and 'trend_pct' excluded from features? ",
      label_source not in used_features and 'trend_pct' not in used_features)

# correlation sanity check — anything suspiciously perfect?
corr_check = df[numeric + ['is_declining_label']].corr()['is_declining_label'].abs().sort_values(ascending=False)
print("\nAbsolute correlation with label (top 5):")
print(corr_check.head(5))

Features used: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'days_with_impressions', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'content_age_days', 'competition_level', 'content_type', 'main_intent']

Label-derivation check: 'trend_direction' and 'trend_pct' excluded from features?  True

Absolute correlation with label (top 5):
is_declining_label        1.000000
days_with_impressions     0.190055
content_age_days          0.163882
word_count                0.090157
days_since_last_update    0.081383
Name: is_declining_label, dtype: float64


## Leakage audit

- `trend_direction` (label source) and `trend_pct` (same underlying signal)
  are confirmed excluded from the feature list.
- No product/CMS flags, client identifiers, or post-hoc outcome fields are
  used as features — client_id is used only for the split, never as a model
  input.
- Correlation check: the highest correlation with the label among features is
  moderate (not near 1.0), which is consistent with a real signal rather than
  a disguised copy of the label. If any feature showed correlation above ~0.9,
  that would be the signal to investigate for hidden leakage.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Claim rewrite

**My boldest original sentence (from Week 5):** "Logistic Regression beat both
the baseline and Random Forest on this split."

**Rewritten in safe language:** "Under a client-grouped holdout split, Logistic
Regression showed a higher *observed* Precision@50 (~0.70) than both the
Week-4 baseline rule (~0.32) and Random Forest (~0.58) on this dataset. This
is a *directional* result from one dataset and one random seed — it should be
read as decision-support for prioritizing review, not as a general claim that
logistic regression outperforms random forest for content-decline prediction."

This softens two things: it names the exact metric/split instead of implying
a universal win, and it flags the result as observed-on-this-data rather than
a general truth about the two algorithms.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.